# Task 1 — Inconsistencies and duplicates

**Daniel Ehab** · 58-3825 · MET-02 · CSEN1095 Data Engineering, W26

Student clubs collected sign-ups through an online form for one week, and `club_signups.csv` holds one row per form submission. This notebook cleans it into a table with **one row per student per club** and every value spelled one way, then saves it as `club_signups_clean.csv`.

The notebook follows the Lab 3 rules:

1. **The raw file is never edited.** It is read once into `raw`, and every fix happens on a copy.
2. **Every fix is code.** Each fix is followed by a ✓ check cell that fails loudly if the fix did not work.
3. **Count before and after.** Every step that changes rows says how many.

## Setup

The course's pinned environment from Lab 1: Python 3.12 and pandas 2.3.3. The versions are printed so that anyone who re-runs the notebook can compare.

In [1]:
import sys

import pandas as pd

print("Python", sys.version.split()[0])
print("pandas", pd.__version__)

Python 3.12.14
pandas 2.3.3


## Part A — Inconsistencies

### A.1 Look first

Before fixing anything, we answer the Lab 3 first-look questions for this file: what one row is, whether the key is unique, what type each column has, what the categories look like, and where the nulls are.

The file is read once into `raw`. Nothing assigns to `raw` after this cell; every fix later works on a copy.

In [2]:
raw = pd.read_csv("club_signups.csv")

print(raw.shape)
raw.head()

(39, 9)


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Soccer,2026-09-15 13:39,N
1,S010,61-3846,AHMED ZAKI,ahmed.zaki@student.guc.edu.eg,Management,alexandria,Chess,15/09/2026 03:12,false
2,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32,Y
3,S035,61-4844,MOHAMED ADEL,mohamed.adel@student.guc.edu.eg,Engineering and Materials Science,Cairo,Debating,2026-09-17 11:15,yes
4,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess Club,2026-09-13 22:36,yes


**Types.** Which type did pandas give each column?

In [3]:
raw.dtypes

signup_id       object
student_id      object
full_name       object
email           object
faculty         object
city            object
club            object
signed_up_at    object
fee_paid        object
dtype: object

**Distinct values and nulls.** How many different values does each column hold, and is anything missing?

In [4]:
pd.DataFrame({"distinct": raw.nunique(), "missing": raw.isna().sum()})

,distinct,missing
signup_id,36,0
student_id,15,0
full_name,24,0
email,18,0
faculty,13,0
city,9,0
club,13,0
signed_up_at,36,0
fee_paid,9,0


**The key.** The form gave every submission a `signup_id`. Is it unique?

In [5]:
print("rows:", len(raw), "| distinct signup_id:", raw["signup_id"].nunique(), "| unique?", raw["signup_id"].is_unique)

rows: 39 | distinct signup_id: 36 | unique? False


**The categories.** `value_counts()` of every categorical column, including any nulls (`dropna=False`).

In [6]:
for col in ["faculty", "club", "city", "fee_paid"]:
    print(raw[col].value_counts(dropna=False), "\n")

faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
iet                                       3
MET                                       3
Engineering and Materials Science         1
Pharma                                    1
Media Engineering and Technology          1
met                                       1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64 

club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Football       3
Chess Club     3
Soccer         2
music          2
chess          2
debate club    1
Music          1
Debate         1
Name: count, dtype: int64 

city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza           2
alexandria     1
CAIRO         

**What we see**, and where each problem gets fixed:

- **One row is one form submission.** 39 rows and 9 columns, as the brief says, and no value is missing anywhere.
- **Every column is text** (`object`). `fee_paid` should be True/False (→ A.4) and `signed_up_at` should be a datetime (→ A.5). The first five rows already show two date shapes: `2026-09-15 13:39` and `15/09/2026 03:12`.
- **The key repeats.** 39 rows but only 36 distinct `signup_id`s, so 3 rows reuse an id that already exists (→ B.1).
- **Too many spellings.** `faculty` has 13 distinct values where 5 are allowed, `club` 13 for 5, and `city` 9 for 3 (→ A.2). The variants come in three kinds: case (`iet`, `CAIRO`), longer or shorter names (`Information Engineering and Technology`, `Pharma`, `Chess Club`, `Alex`), and synonyms (`Soccer`, `Debating`).
- **A value that prints twice.** `Music` appears twice in the `club` counts (5 and 1). Two values that print the same must differ in something invisible (→ checked in A.2).
- **One person, several spellings.** 15 distinct `student_id`s but 24 distinct names and 18 distinct emails, e.g. `AHMED ZAKI` and `MOHAMED ADEL` in capitals (→ A.3).
- **Nine ways to write yes or no** in `fee_paid`: `1`, `yes`, `Y`, `TRUE`, `Yes`, `false`, `N`, `0`, `no` (→ A.4).